# Uplift Modeling em Marketing — Notebook 5: Avaliação Confirmatória no Teste Selado (S6)

> Continuação de `03_Meta_Learners_PT.ipynb` e `04_Causal_Forest_Uplift_Trees_PT.ipynb`.
> Este notebook não compartilha o kernel dos anteriores — recarrega o que
> precisar a partir de `src/` e dos artefatos persistidos em `artifacts/s6/`.
> Os Notebooks 03 e 04 permanecem congelados: nenhuma célula ali foi alterada
> nesta rodada.
>
> **Objetivo do Notebook 5 (S6):** avaliação confirmatória, uma única vez, do
> modelo primário pré-selecionado (`UpliftTree`) e de três comparadores
> pré-especificados (X+Tree, S+LightGBM, baseline de propensão) no teste
> selado — a partição de 20% do dataset nunca usada em nenhuma decisão de
> modelagem até aqui. Toda decisão relevante (modelo primário, comparadores,
> métricas, critério de interpretação, protocolo de bootstrap, modelos já
> treinados e congelados) foi tomada e registrada **antes** da abertura do
> teste — ver `artifacts/s6/preregistration.json` e
> `artifacts/s6/final_models.joblib`.
>
> **O teste selado foi aberto uma única vez nesta execução autorizada**,
> após pré-registro e congelamento dos modelos (ver 6.1/6.2). As seções
> 6.4–6.6 registram a avaliação confirmatória executada — modelo primário
> e comparadores pontuados uma única vez, métricas e bootstrap
> calculados, resultados persistidos. O sentinel
> `SEALED_TEST_EVALUATED.json`, criado ao final dessa execução, impede
> qualquer nova execução da avaliação.


## Índice

- [Seção 6 — Avaliação Confirmatória no Teste Selado](#s6)
    - [6.1 Pré-registro e protocolo congelado](#s6-1)
    - [6.2 Modelos finais congelados](#s6-2)
    - [6.3 Guarda irreversível](#s6-3)
    - [6.4 Avaliação confirmatória](#s6-4)
    - [6.5 Resultado da hipótese primária](#s6-5)
    - [6.6 Comparações secundárias](#s6-6)
    - [6.7 Síntese final](#s6-7)

---

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import joblib
import pandas as pd

# Bootstrap de path: permite `from src...` a partir de notebooks/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.i18n import make_lang
from src.viz import apply_plot_style

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

lang = make_lang('pt')
apply_plot_style()

# Caminhos dos artefatos de S6 (todos gerados ANTES da abertura do teste selado)
ARTIFACTS_S6_DIR = PROJECT_ROOT / 'artifacts' / 's6'
PREREG_PATH = ARTIFACTS_S6_DIR / 'preregistration.json'
FINAL_MODELS_PATH = ARTIFACTS_S6_DIR / 'final_models.joblib'
SEALED_TEST_SCORES_PATH = ARTIFACTS_S6_DIR / 'sealed_test_scores.parquet'
S6_RESULTS_PATH = ARTIFACTS_S6_DIR / 's6_results.json'
SENTINEL_PATH = ARTIFACTS_S6_DIR / 'SEALED_TEST_EVALUATED.json'


<a id='s6'></a>
<a id="s6"></a>

## Seção 6 — Avaliação Confirmatória no Teste Selado

S4 e S5 avaliaram sete candidatos ao todo, sempre em `train_df`/`val_df` —
nunca no teste selado. Ao final de S5 (Notebook 04, seção 5.8), essa
avaliação de desenvolvimento produziu uma decisão registrada: `UpliftTree`
como modelo primário, com X+Tree(depth=4), S+LightGBM(vanilla) e o baseline
de propensão como comparadores pré-especificados. Esta seção executa —
quando autorizada — a única avaliação confirmatória do projeto: abrir o
teste selado uma única vez, pontuar os quatro modelos já congelados (sem
reajuste de hiperparâmetro nenhum) e testar a hipótese primária
pré-registrada com um intervalo de confiança bootstrap calculado antes de
qualquer resultado ser observado.


<a id="s6-1"></a>

### 6.1 Pré-registro e protocolo congelado

Carrega e imprime `artifacts/s6/preregistration.json` — a especificação
completa registrada antes da abertura do teste: estimand, modelo primário,
comparadores, hipótese confirmatória primária, métricas, protocolo de
bootstrap e as diretrizes absolutas desta rodada. O SHA-256 do arquivo é
calculado e reportado para conferência de integridade.


In [2]:
prereg = json.loads(PREREG_PATH.read_text(encoding='utf-8'))
prereg_sha256 = hashlib.sha256(PREREG_PATH.read_bytes()).hexdigest()

assert prereg['status'] == 'preregistered_before_sealed_test'
assert prereg['created_before_test_unlock'] is True

labels = lang({'header': 'Pré-registro de S6 (artifacts/s6/preregistration.json)'})
print(f"{labels['header']}:\n")
print(f"  stage:                 {prereg['stage']}")
print(f"  status:                {prereg['status']}")
print(f"  estimand:              {prereg['estimand']}")
print(f"  primary_outcome:       {prereg['primary_outcome']}")
print(f"  primary_model:         {prereg['primary_model']}")
print(f"  comparators:           {prereg['comparators']}")
print(f"  excluded_from_s6:      {prereg['excluded_from_s6']['models']}")
print(f"  primary_hypothesis:    {prereg['primary_hypothesis']}")
print(f"  primary_metric:        {prereg['primary_metric']}")
print(f"  secondary_metrics:     {prereg['secondary_metrics']}")
print(f"  primary_comparison:    {prereg['primary_comparison']}")
print(f"  secondary_comparisons: {prereg['secondary_comparisons']}")
print(f"  bootstrap n_boot/seed: {prereg['bootstrap_protocol']['n_boot']} / {prereg['bootstrap_protocol']['seed']}")
print(f"  feature_cols:          {prereg['feature_cols']}")
print(f"  seed:                  {prereg['seed']}")
print(f"  created_before_test_unlock: {prereg['created_before_test_unlock']}")
print(f"\n  SHA-256 do arquivo: {prereg_sha256}")


Pré-registro de S6 (artifacts/s6/preregistration.json):

  stage:                 S6
  status:                preregistered_before_sealed_test
  estimand:              any email vs. No E-Mail (treatment pooled binário, RCT — three-arm pooled to two)
  primary_outcome:       visit
  primary_model:         UpliftTree
  comparators:           ['X+Tree(depth=4)', 'S+LightGBM(vanilla)', 'Baseline (propensão)']
  excluded_from_s6:      ['UpliftRF', 'CausalForest']
  primary_hypothesis:    UpliftTree produz ranking incremental melhor que o baseline de propensão de resposta no teste selado.
  primary_metric:        qini_auc (normalized Qini AUC via sklift.metrics.qini_auc_score)
  secondary_metrics:     ['uplift_auc', 'uplift_at_30pct']
  primary_comparison:    UpliftTree - Baseline (propensão)
  secondary_comparisons: ['UpliftTree - X+Tree(depth=4)', 'UpliftTree - S+LightGBM(vanilla)', 'X+Tree(depth=4) - Baseline (propensão)', 'S+LightGBM(vanilla) - Baseline (propensão)']
  bootstrap n_boot/s

<a id="s6-2"></a>

### 6.2 Modelos finais congelados

Carrega `artifacts/s6/final_models.joblib` — os quatro modelos treinados uma
única vez em `dev_df = train_df + val_df` (80% de desenvolvimento, nenhuma
linha do teste selado), com a metadata gravada junto ao artefato. Confirma
que o hash do pré-registro embutido na metadata bate exatamente com o hash
calculado em 6.1 acima — uma checagem de integridade entre os dois
artefatos, não uma reabertura de nada sensível.


In [3]:
bundle = joblib.load(FINAL_MODELS_PATH)
models = bundle['models']
encoder = bundle['encoder']
metadata = bundle['metadata']

assert metadata['preregistration_sha256'] == prereg_sha256, (
    'Hash do pré-registro embutido em final_models.joblib não bate com o '
    'preregistration.json atual -- os modelos podem ter sido congelados sob '
    'uma especificação diferente da vigente.'
)

labels = lang({'header': 'Modelos finais congelados (artifacts/s6/final_models.joblib)'})
print(f"{labels['header']}:\n")
for name in metadata['model_names']:
    print(f"  {name:22s} -> {type(models[name]).__name__:28s} hyperparams={metadata['hyperparameters'][name]}")
print(f"\n  n_train (dev): {metadata['n_train']}")
print(f"  n_val   (dev): {metadata['n_val']}")
print(f"  n_dev total:   {metadata['n_dev']}  ({metadata['training_data']})")
print(f"  seed:          {metadata['seed']}")
print(f"  package_versions: {metadata['package_versions']}")
print(f"  created_before_test_unlock: {metadata['created_before_test_unlock']}")
print(f"  timestamp_utc: {metadata['timestamp_utc']}")
print('\n  Integridade preregistration_sha256 <-> preregistration.json: OK (hash bate)')


Failed to import duecredit due to No module named 'duecredit'


Modelos finais congelados (artifacts/s6/final_models.joblib):

  UpliftTree             -> UpliftTreeClassifier         hyperparams={'control_name': 'control', 'random_state': 42}
  X+Tree(depth=4)        -> BaseXRegressor               hyperparams={'base_learner': 'DecisionTreeRegressor', 'max_depth': 4, 'random_state': 42}
  S+LightGBM(vanilla)    -> BaseSRegressor               hyperparams={'base_learner': 'LGBMRegressor', 'random_state': 42}
  Baseline (propensão)   -> LGBMClassifier               hyperparams={'base_learner': 'LGBMClassifier', 'random_state': 42, 'trained_on': 'só linhas tratadas de dev_df'}

  n_train (dev): 38400
  n_val   (dev): 12800
  n_dev total:   51200  (dev_df = train_df + val_df (nenhuma linha do teste selado))
  seed:          42
  package_versions: {'causalml': '0.15.5', 'lightgbm': '4.7.0', 'scikit-learn': '1.6.1', 'scikit-uplift': '0.5.1', 'numpy': '2.4.6', 'pandas': '2.3.3'}
  created_before_test_unlock: True
  timestamp_utc: 2026-08-11T19:43:39.0904

<a id="s6-3"></a>

### 6.3 Guarda irreversível

O teste selado só pode ser aberto com `unlock=True` explícito em
`load_sealed_test` (ver `src/splits.py`). A célula abaixo é uma segunda
barreira, local a este notebook: enquanto `UNLOCK_SEALED_TEST`
permanecesse `False`, a execução seria interrompida antes de qualquer
célula que pudesse acessar o teste. `UNLOCK_SEALED_TEST` permaneceu
`False` durante toda a fase de preparação (pré-registro, congelamento
dos modelos, pré-flight de hashes). Após autorização explícita e
específica, a guarda foi alterada para `True` e a célula abaixo foi
executada uma única vez, abrindo caminho para a avaliação confirmatória
de 6.4. **`UNLOCK_SEALED_TEST = True` permanece no código como registro
histórico dessa execução autorizada** — não é revertido para `False`
retroativamente. O estado pós-execução é protegido pelo sentinel
`artifacts/s6/SEALED_TEST_EVALUATED.json` (ver 6.4): qualquer tentativa
de reexecutar a célula de avaliação confirmatória é interrompida
automaticamente, sem reabrir o teste nem recalcular nada.


In [4]:
UNLOCK_SEALED_TEST = True  # Autorização explícita recebida -- avaliação confirmatória de S6

if not UNLOCK_SEALED_TEST:
    raise RuntimeError(
        "Teste selado permanece fechado. Pré-registro e modelos finais estão "
        "congelados (ver 6.1/6.2). Alterar UNLOCK_SEALED_TEST para True "
        "somente após autorização explícita, em uma rodada dedicada à "
        "avaliação confirmatória de S6."
    )


<a id="s6-4"></a>

### 6.4 Avaliação confirmatória

**Executada uma única vez após autorização explícita**, com
`UNLOCK_SEALED_TEST = True` em 6.3. A célula verificou primeiro que
`artifacts/s6/SEALED_TEST_EVALUATED.json` ainda não existia (proteção
contra reexecução silenciosa — sem `force=True`); em seguida carregou os
quatro modelos já congelados de `final_models.joblib`, abriu `test_df`
uma única vez via `load_sealed_test(df_pooled, unlock=True)`, gerou os
scores dos quatro modelos (nenhum reajuste de hiperparâmetro ou de
modelo), calculou Qini AUC/Uplift AUC/Uplift@30% absolutos dos quatro, e
rodou o bootstrap pareado pré-especificado (`bootstrap_qini_comparison`,
2000 réplicas, estratificado por braço) para Qini AUC e os cinco deltas
de 6.5/6.6. Os resultados foram persistidos em
`sealed_test_scores.parquet` + `s6_results.json`, e o sentinel
`SEALED_TEST_EVALUATED.json` foi criado ao final, bloqueando qualquer
nova execução desta célula.


In [5]:
from datetime import datetime, timezone

from src.config import POOLED_TREATMENT_COL, PRIMARY_OUTCOME, TREATMENT_COL
from src.data import add_pooled_treatment, load_hillstrom
from src.evaluation import bootstrap_qini_comparison, evaluate_multiple_rankings
from src.learners import (
    encode_meta_learner_features, predict_propensity_score,
    predict_single_meta_learner, predict_uplift_tree_uplift,
)
from src.splits import dataset_fingerprint, load_sealed_test

if SENTINEL_PATH.exists():
    raise RuntimeError(
        f"{SENTINEL_PATH} já existe -- S6 já foi executada. Esta célula não "
        "reabre o teste selado nem recalcula métricas silenciosamente. "
        "Leia s6_results.json para o resultado já produzido; não há opção "
        "force=True para contornar esta proteção."
    )

df = load_hillstrom()
df_pooled = add_pooled_treatment(df)

test_df = load_sealed_test(df_pooled, unlock=True)  # única abertura do teste selado deste projeto

X_test = encode_meta_learner_features(test_df, encoder)
scores_test = {
    'UpliftTree': predict_uplift_tree_uplift(models['UpliftTree'], X_test),
    'X+Tree(depth=4)': predict_single_meta_learner('X', models['X+Tree(depth=4)'], X_test),
    'S+LightGBM(vanilla)': predict_single_meta_learner('S', models['S+LightGBM(vanilla)'], X_test),
    'Baseline (propensão)': predict_propensity_score(models['Baseline (propensão)'], test_df),
}

y_test = test_df[PRIMARY_OUTCOME].to_numpy(dtype=float)
treatment_test = test_df[POOLED_TREATMENT_COL].to_numpy()
arm_test = test_df[TREATMENT_COL].to_numpy()

metrics_table = evaluate_multiple_rankings(y_test, scores_test, treatment_test)
labels = lang({'header': 'Métricas absolutas no teste selado (única avaliação)'})
print(f"{labels['header']}:")
print(metrics_table.round(4))

deltas_spec = [
    ('UpliftTree', 'Baseline (propensão)'),
    ('UpliftTree', 'X+Tree(depth=4)'),
    ('UpliftTree', 'S+LightGBM(vanilla)'),
    ('X+Tree(depth=4)', 'Baseline (propensão)'),
    ('S+LightGBM(vanilla)', 'Baseline (propensão)'),
]
bootstrap_result = bootstrap_qini_comparison(
    y_test, treatment_test, arm_test, scores_test, deltas_spec,
    n_boot=prereg['bootstrap_protocol']['n_boot'], seed=prereg['bootstrap_protocol']['seed'],
)

scores_out = pd.DataFrame({
    'row_index': test_df.index.values,
    'score_uplift_tree': scores_test['UpliftTree'],
    'score_x_tree': scores_test['X+Tree(depth=4)'],
    'score_s_lgbm': scores_test['S+LightGBM(vanilla)'],
    'score_response_baseline': scores_test['Baseline (propensão)'],
})
scores_out.to_parquet(SEALED_TEST_SCORES_PATH, index=False)

final_models_sha256 = hashlib.sha256(FINAL_MODELS_PATH.read_bytes()).hexdigest()
results = {
    'metrics_absolute': metrics_table.round(6).to_dict(orient='index'),
    'bootstrap': bootstrap_result,
    'preregistration_sha256': prereg_sha256,
    'final_models_sha256': final_models_sha256,
    'dataset_fingerprint': dataset_fingerprint(),
    'n_test': int(len(test_df)),
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
S6_RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')

SENTINEL_PATH.write_text(json.dumps({
    'evaluated': True,
    'timestamp_utc': results['timestamp_utc'],
    's6_results_path': str(S6_RESULTS_PATH),
}, indent=2), encoding='utf-8')

print(f"\nSalvos: {SEALED_TEST_SCORES_PATH.name}, {S6_RESULTS_PATH.name}, {SENTINEL_PATH.name}")


Métricas absolutas no teste selado (única avaliação):
                      qini_auc  uplift_auc  uplift_at_30pct
UpliftTree              0.0089      0.0050           0.0567
X+Tree(depth=4)         0.0470      0.0278           0.0866
S+LightGBM(vanilla)     0.0175      0.0107           0.0666
Baseline (propensão)    0.0177      0.0103           0.0883



Salvos: sealed_test_scores.parquet, s6_results.json, SEALED_TEST_EVALUATED.json


<a id="s6-5"></a>

### 6.5 Resultado da hipótese primária

**Executada uma única vez, na sequência de 6.4.** Aplicou
automaticamente a regra de interpretação já pré-registrada em
`preregistration.json` (`interpretation_rule_primary`) ao delta primário
observado ΔQini = Qini(UpliftTree) − Qini(baseline de propensão), sem
alterar modelo ou hiperparâmetro nesta célula — ver o resultado e o
veredito automático no output abaixo, e a leitura completa em 6.7.


In [6]:
delta_primary = bootstrap_result['deltas'][prereg['primary_comparison']]
print(f"ΔQini primário ({prereg['primary_comparison']}) = {delta_primary['point_estimate']:.4f}")
print(f"IC 95% bootstrap: [{delta_primary['ci_low']:.4f}, {delta_primary['ci_high']:.4f}]")

rule = prereg['interpretation_rule_primary']
if delta_primary['ci_low'] > 0:
    veredito = rule['ci_entirely_above_zero']
elif delta_primary['ci_high'] < 0:
    veredito = rule['ci_entirely_below_zero']
else:
    veredito = rule['ci_includes_zero']

print(f"\nVeredito (regra pré-registrada, fixada antes da abertura do teste):\n  {veredito}")


ΔQini primário (UpliftTree - Baseline (propensão)) = -0.0088
IC 95% bootstrap: [-0.0492, 0.0302]

Veredito (regra pré-registrada, fixada antes da abertura do teste):
  não há evidência confirmatória de vantagem


<a id="s6-6"></a>

### 6.6 Comparações secundárias

**Executada uma única vez, na sequência de 6.4.** Mostrou os quatro
deltas secundários pré-especificados e os valores absolutos dos quatro
candidatos, sem escolher um novo modelo com base neles — ver o output
abaixo, e a leitura completa em 6.7.


In [7]:
for key in prereg['secondary_comparisons']:
    d = bootstrap_result['deltas'][key]
    print(f"{key:45s} Δ={d['point_estimate']:+.4f}  IC95%=[{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]")

labels = lang({'header': 'Valores absolutos dos quatro candidatos no teste selado'})
print(f"\n{labels['header']}:")
print(metrics_table.round(4))

print(
    "\nEssas comparações são pré-especificadas e reportadas por completude "
    "(inclusive para encerrar a pergunta de S4 sobre X+Tree/S+LightGBM vs. "
    "baseline), mas não substituem retrospectivamente o modelo primário nem "
    "elegem um novo 'vencedor' com base no maior Qini absoluto observado aqui."
)


UpliftTree - X+Tree(depth=4)                  Δ=-0.0381  IC95%=[-0.0682, -0.0097]
UpliftTree - S+LightGBM(vanilla)              Δ=-0.0086  IC95%=[-0.0412, +0.0220]
X+Tree(depth=4) - Baseline (propensão)        Δ=+0.0293  IC95%=[-0.0082, +0.0637]
S+LightGBM(vanilla) - Baseline (propensão)    Δ=-0.0003  IC95%=[-0.0295, +0.0309]

Valores absolutos dos quatro candidatos no teste selado:
                      qini_auc  uplift_auc  uplift_at_30pct
UpliftTree              0.0089      0.0050           0.0567
X+Tree(depth=4)         0.0470      0.0278           0.0866
S+LightGBM(vanilla)     0.0175      0.0107           0.0666
Baseline (propensão)    0.0177      0.0103           0.0883

Essas comparações são pré-especificadas e reportadas por completude (inclusive para encerrar a pergunta de S4 sobre X+Tree/S+LightGBM vs. baseline), mas não substituem retrospectivamente o modelo primário nem elegem um novo 'vencedor' com base no maior Qini absoluto observado aqui.


<a id="s6-7"></a>

### 6.7 Síntese final

**Resultado da hipótese confirmatória primária.** No teste selado (n=12.800,
20% do dataset, nunca antes acessado), ΔQini = Qini(UpliftTree) −
Qini(Baseline de propensão) = −0,0088, IC 95% bootstrap pareado e
estratificado pelos três braços originais (2.000 réplicas) = [−0,0492,
+0,0302]. Como o intervalo contém zero, a regra de interpretação
pré-registrada em `preregistration.json` aplica o veredito **"não há
evidência confirmatória de vantagem"**: a hipótese primária — UpliftTree
produzir ranking incremental melhor que o baseline de propensão — não foi
confirmada nesta amostra selada. O ponto estimado é negativo, mas o IC não
fica inteiramente abaixo de zero, então tampouco há evidência confirmatória
de que o baseline supere o UpliftTree — o resultado é inconclusivo por este
critério, não uma derrota estatisticamente distinguível.

**Resultados secundários.** Os quatro deltas pré-especificados, todos via o
mesmo bootstrap pareado:

| Comparação | Δ | IC 95% |
|---|---|---|
| UpliftTree − X+Tree(depth=4) | −0,0381 | [−0,0682, −0,0097] |
| UpliftTree − S+LightGBM(vanilla) | −0,0086 | [−0,0412, +0,0220] |
| X+Tree(depth=4) − Baseline (propensão) | +0,0293 | [−0,0082, +0,0637] |
| S+LightGBM(vanilla) − Baseline (propensão) | −0,0003 | [−0,0295, +0,0309] |

Valores absolutos de Qini AUC: UpliftTree 0,0089; X+Tree(depth=4) 0,0470;
S+LightGBM(vanilla) 0,0175; Baseline (propensão) 0,0177. O IC 95%
bootstrap da comparação secundária pré-especificada UpliftTree −
X+Tree(depth=4) excluiu zero em favor de X+Tree ([−0,0682, −0,0097]).
Como comparação secundária, esse IC não recebeu ajuste por
multiplicidade e não substitui a hipótese confirmatória primária. Apesar
de X+Tree(depth=4) ter o maior Qini absoluto entre os quatro candidatos,
seu delta contra o baseline inclui zero (por pouco: IC inferior
−0,0082) — ou seja, essa vantagem não é distinguível do zero por este
critério. O delta de S+LightGBM(vanilla) contra o baseline é
essencialmente nulo (−0,0003), fechando a pergunta deixada em aberto em S4:
no teste selado, S+LightGBM não mostra vantagem distinguível sobre o
baseline. Como pré-especificado, nenhuma dessas comparações substitui
retroativamente o modelo primário nem elege um novo "vencedor" — UpliftTree
permanece o modelo primário pré-registrado, independentemente do Qini
absoluto observado aqui para os demais candidatos.

**Generalização ou não do padrão visto em desenvolvimento.** Em
desenvolvimento (S4/S5, repeated holdout), UpliftTree teve a maior média de
Qini (0,0254) e a maior taxa de vitória (40%) entre os seis candidatos
avaliados, batendo o baseline em 10 de 15 reamostragens — o principal
argumento por trás de sua escolha como modelo primário (Notebook 04, 5.8).
**Esse padrão não se reproduziu no teste selado:** UpliftTree teve o pior
Qini absoluto dos quatro candidatos avaliados aqui (0,0089), abaixo
inclusive do seu já fraco resultado no único holdout fixo de S5 (0,0105) —
resultado que já havia sido registrado em 5.7 como um "padrão de
sensibilidade a protocolo/amostra", e que se mostrou, neste caso, mais
informativo sobre o teste selado do que a média do protocolo repetido. Por
outro lado, X+Tree(depth=4) — que em desenvolvimento estava em "empate
descritivo" com UpliftTree (Δ médio +0,0004, 8/15 splits) — teve, de longe,
o melhor desempenho absoluto no teste selado, embora sem separação
distinguível do baseline pelo IC bootstrap. Em suma: o teste selado não
confirma, para o candidato pré-selecionado como primário, o padrão
observado no protocolo de desenvolvimento — isso é, em si, um resultado
válido e informativo sobre os limites do critério de seleção usado
(repeated holdout parcialmente sobreposto, ver 5.7), não um erro de
execução desta avaliação.

**Limitações.**

- O teste selado é uma única amostra (n=12.800); os ICs bootstrap refletem
  a incerteza de reamostragem dentro **dessa** amostra, não a variabilidade
  entre múltiplas amostras selado possíveis — diferente do repeated holdout
  de desenvolvimento, que reamostra repetidamente o mesmo `train_df`.
- Os quatro modelos foram pontuados uma única vez, sem re-treino, exatamente
  como planejado — não há, com os dados desta rodada, como distinguir se o
  desempenho fraco do UpliftTree no teste é ruído específico desta partição
  de 20% ou um sinal mais genérico de que o critério de seleção do
  protocolo de desenvolvimento não generaliza bem para esse candidato.
- Cinco deltas foram calculados (1 primário + 4 secundários) sem correção
  para comparações múltiplas nos ICs individuais — consistente com o
  pré-registro, que trata apenas a comparação primária como confirmatória e
  as demais como descritivas.
- Por desenho (guardrails pré-registrados), esta avaliação não pode ser
  reaberta nem repetida — o sentinel `SEALED_TEST_EVALUATED.json` bloqueia
  qualquer nova execução da seção 6.4 sem `force=True` (que não existe).
  Isso significa que nenhuma tentativa de replicação independente é
  possível dentro deste projeto.
- Este resultado não deve ser usado para reselecionar um modelo "vencedor"
  nem para justificar ajustes retroativos de hiperparâmetro, protocolo ou
  features — nenhuma dessas ações foi tomada nesta rodada, e a hipótese
  primária permanece definida exatamente como registrada antes da abertura
  do teste.

**Estado ao final de S6:** a hipótese confirmatória primária (UpliftTree
vs. baseline de propensão) não foi confirmada nesta amostra selada. Nenhum
novo modelo foi eleito como sucessor do UpliftTree. O projeto encerra a
fase de avaliação confirmatória com um resultado negativo para a hipótese
principal — registrado como tal, sem reinterpretação a posteriori.
